# Load London Transport

### Estimated time: 5-10 minutes

This notebook loads the complete London Transport Network into Neo4j in a single step:

1. **CSV to Delta Lake** - Reads station and tube line data from the Unity Catalog Volume into Delta tables
2. **Station nodes** - Writes 302 Station nodes to Neo4j via the Spark Connector
3. **Tube line relationships** - Creates bidirectional relationships with **line-specific types** (e.g., `:BAKERLOO`, `:CENTRAL`, `:CIRCLE`)

After this notebook runs, Neo4j contains the complete graph:
- **302 Station nodes** with names, zones, postcodes, and coordinates
- **Bidirectional relationships** for all tube lines using line-specific relationship types

### Prerequisites

- Run **0 - Required Setup** first
- Cluster must be **Dedicated** mode with the Neo4j Spark Connector Maven library installed

## Data Model

The London Transport graph uses a simple but effective model:

```
(:Station) -[:BAKERLOO]->    (:Station)
(:Station) -[:CENTRAL]->     (:Station)
(:Station) -[:CIRCLE]->      (:Station)
(:Station) -[:DISTRICT]->    (:Station)
(:Station) -[:JUBILEE]->     (:Station)
(:Station) -[:METROPOLITAN]->(:Station)
(:Station) -[:NORTHERN]->    (:Station)
(:Station) -[:PICCADILLY]->  (:Station)
(:Station) -[:VICTORIA]->    (:Station)
...and more
```

**Key design decision:** Each tube line is a **relationship type**, not a property on a
generic `:CONNECTED` relationship. This provides better query performance, clearer
semantics, and better visualization in Neo4j Browser.

## Run the Import

In [ ]:
%run ./Includes/_lib/london_transport_import

In [ ]:
run_full_import()

## Explore the Graph

Open Neo4j Browser and try these queries:

**See the full schema:**
```cypher
CALL db.schema.visualization()
```

**Count stations:**
```cypher
MATCH (s:Station)
RETURN count(s) AS total_stations
```

**Connections by tube line:**
```cypher
MATCH ()-[r]->()
RETURN type(r) AS line, count(r) AS connections
ORDER BY connections DESC
```

**Top 10 busiest stations:**
```cypher
MATCH (s:Station)
RETURN s.name AS station, count{(s)-[]-()} AS total_connections
ORDER BY total_connections DESC
LIMIT 10
```

**Baker Street connections (all lines):**
```cypher
MATCH (s:Station {name: 'Baker Street'})-[r]-(connected:Station)
RETURN s.name AS from_station, type(r) AS tube_line, connected.name AS to_station
```

**Shortest path between two stations:**
```cypher
MATCH path = shortestPath(
  (from:Station {name: "King's Cross St. Pancras"})-[*..5]-(to:Station {name: 'Victoria'})
)
RETURN path, length(path) AS hops,
       [rel IN relationships(path) | type(rel)] AS lines_used
```

## Next Steps

The graph is fully loaded. Continue to:

- **2 - Query London Transport**: Ask questions in natural language using text-to-Cypher
- **Neo4j Browser**: Explore the graph visually